<a href="https://colab.research.google.com/github/xiaofanpaiooo-code/whisper-lora/blob/main/%E5%9F%BA%E4%BA%8E%E6%B7%B1%E5%BA%A6%E5%AD%A6%E4%B9%A0%E7%9A%84%E6%99%BA%E8%83%BD%E8%AF%AD%E9%9F%B3%E5%AD%97%E5%B9%95%E7%B3%BB%E7%BB%9F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Cell 1] 环境准备与模型加载

In [1]:
!pip install datasets transformers librosa soundfile

In [2]:
# 确保已安装必要的库：!pip install datasets transformers librosa soundfile
from transformers import WhisperProcessor
from datasets import load_dataset, interleave_datasets, Audio

# 加载特征提取器与分词器 (以 whisper-small 为例)
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="Chinese",
    task="transcribe"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

[Cell 2] 数据处理

In [6]:
from datasets import load_dataset, interleave_datasets, Audio

# 1. 开启流式加载，彻底规避 OOM
ds_65h = load_dataset("gongqingyu/bishe_whisper_dataset_65h", split="train", streaming=True)
ds_35h = load_dataset("gongqingyu/bishe_whisper_dataset_35h2", split="train", streaming=True)

# ================= 核心修复：合并前强制对齐特征 Schema =================
# 步骤 A：强行将两个流的音频特征空间锚定为 16000Hz（Whisper的标准输入频率）
ds_65h = ds_65h.cast_column("audio", Audio(sampling_rate=16000))
ds_35h = ds_35h.cast_column("audio", Audio(sampling_rate=16000))

# 步骤 B：文本标签列名对齐 (⚠️ 强烈预警：Schema 必须100%一致)
# 假设你在构建数据时，ds_65h 的文本叫 "text"，而 ds_35h 叫 "transcription"
# 这里提供一个健壮的预处理，先检查列名并统一改名为 "transcription"
def align_text_column(dataset):
    column_names = list(dataset.features.keys())
    if "text" in column_names and "transcription" not in column_names:
        return dataset.rename_column("text", "transcription")
    elif "sentence" in column_names and "transcription" not in column_names:
        return dataset.rename_column("sentence", "transcription")
    return dataset

ds_65h = align_text_column(ds_65h)
ds_35h = align_text_column(ds_35h)

# ================= 移除无关的冗余列 (精简内存管线) =================
# 不同数据集可能带有特有的额外字段（如 client_id, up_votes 等），这些会导致 Schema 依然不匹配
# 我们只保留 ASR 所需的核心列：'audio' 和 'transcription'
columns_to_keep = ["audio", "transcription"]
ds_65h = ds_65h.select_columns(columns_to_keep)
ds_35h = ds_35h.select_columns(columns_to_keep)
# ====================================================================

# 2. 动态交替混合数据集 (此时 Schema 已处于绝对安全的强制一致状态)
mixed_dataset = interleave_datasets([ds_65h, ds_35h], probabilities=[0.65, 0.35], seed=42)

# 3. 局部缓冲区打乱
# 缓冲区设为1000，保障每个 mini-batch 存在领域数据与通用数据的黄金混合比
shuffled_dataset = mixed_dataset.shuffle(seed=42, buffer_size=1000)

print("✅ 流式数据集混合与预处理流图已成功重构并完成 Schema 对齐！")

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

✅ 流式数据集混合与预处理流图已成功重构并完成 Schema 对齐！


[Cell 3]定义特征映射管线

In [9]:
def prepare_dataset(batch):
    # 1. 批次解包：此时 batch["audio"] 是一个 list，里面包含多个音频字典
    # 使用列表推导式提取出当前 batch (16条) 所有的音频一维矩阵 array
    audio_arrays = [audio["array"] for audio in batch["audio"]]

    # 2. 批次特征提取：WhisperProcessor 天生支持传入 List[Array] 进行并行计算
    # 因为我们在 Cell 2 已经统一对齐为 16000Hz，这里可以直接写死 16000 节省提取开销
    extracted_features = processor.feature_extractor(
        audio_arrays,
        sampling_rate=16000
    )
    # 直接赋值整个批次的 input_features
    batch["input_features"] = extracted_features.input_features

    # 3. 批次文本分词：同样提取批次文本列表，并行 Tokenize
    text_strings = batch["transcription"]
    tokenized_labels = processor.tokenizer(text_strings)
    batch["labels"] = tokenized_labels.input_ids

    return batch

# 将向量化的映射函数应用到流式数据集中
# batched=True 保障了底层的 C++ 多线程吞吐，最大化规避 OOM 并缩短预处理时间
vectorized_ds = shuffled_dataset.map(prepare_dataset, batched=True, batch_size=16)

print("✅ 批次化特征映射函数已重新挂载！")

✅ 批次化特征映射函数已重新挂载！


[Cell 4] 特征维度验证

In [10]:
# 将 IterableDataset 转为迭代器，抓取第一个样本
sample_iterator = iter(vectorized_ds)
sample = next(sample_iterator)

# 验证核心特征
input_features = sample["input_features"]
labels = sample["labels"]

print("=== 特征映射维度验证报告 ===")
# 预期输出必须严格为: 80 x 3000
print(f"[声学特征] Log-Mel Spectrogram 维度: {len(input_features)} x {len(input_features[0])}")
print(f"[文本特征] Token IDs 长度: {len(labels)}")
print(f"[文本采样] 前 5 个 Token IDs: {labels[:5]}")
print("=============================")

=== 特征映射维度验证报告 ===
[声学特征] Log-Mel Spectrogram 维度: 80 x 3000
[文本特征] Token IDs 长度: 26
[文本采样] 前 5 个 Token IDs: [50258, 50260, 50359, 50363, 3322]
